# Extracting embeddings from a model


# Loading DGCI embeddings:

In [ ]:
import numpy as np                                          # already imported in cell 1

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/latent_codes.npy"
enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 keeps ward's sums stable

In [ ]:
import numpy as np

Z = np.load("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/latent_codes.npy")   # shape: (N, 128)

print("Shape:", Z.shape)

std = np.std(Z, axis=0)
var = np.var(Z, axis=0)

# Sort dimensions by standard deviation
ranking = np.argsort(std)[::-1]

print("\nDimensions ranked by STD:")
print("-" * 50)

for rank, dim in enumerate(ranking):
    print(
        f"Rank {rank+1:3d} | "
        f"Dim {dim:3d} | "
        f"STD = {std[dim]:.8f} | "
        f"Var = {var[dim]:.10e}"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(np.arange(128), std)

plt.xlabel("Latent dimension")
plt.ylabel("Standard deviation")
plt.title("Standard deviation of latent dimensions")

plt.tight_layout()
plt.show()

In [ ]:
std = np.std(Z, axis=0)

print("Minimum STD:", std.min())
print("Maximum STD:", std.max())
print("Median STD:", np.median(std))

print("\nPercentiles:")
for p in [0, 10, 25, 50, 75, 90, 95, 99, 100]:
    print(f"{p:3d}% : {np.percentile(std, p):.8f}")

In [ ]:
for dim in range(128):

    x = Z[:, dim]

    print(
        f"Dim {dim:3d} | "
        f"STD={np.std(x):.7f} | "
        f"MIN={np.min(x):.7f} | "
        f"MAX={np.max(x):.7f} | "
        f"RANGE={np.ptp(x):.7f} | "
        f"UNIQUE={len(np.unique(x))}"
    )

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA



# Standardize latent dimensions
Z_scaled = StandardScaler().fit_transform(Z)

# PCA
pca = PCA()
Z_pca = pca.fit_transform(Z_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

for k in [5, 10, 15, 20, 30, 40, 50, 64, 80, 100, 128]:
    print(
        f"{k:3d} components -> "
        f"{cumulative[k-1]*100:.2f}% variance"
    )

# 5. Let's identify the active dimensions objectively

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


std = np.std(Z, axis=0)

# Sort from largest to smallest
sorted_std = np.sort(std)[::-1]

plt.figure(figsize=(10, 5))
plt.plot(
    np.arange(1, len(sorted_std) + 1),
    sorted_std,
    marker="o"
)

plt.xlabel("Rank")
plt.ylabel("Standard deviation")
plt.title("Sorted latent-dimension standard deviations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    np.arange(1, len(sorted_std) + 1),
    sorted_std,
    marker="o"
)

plt.yscale("log")
plt.xlabel("Rank")
plt.ylabel("Standard deviation (log scale)")
plt.title("Sorted latent-dimension standard deviations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 6. Also calculate the ratio between neighboring dimensions

In [ ]:
sorted_std = np.sort(std)[::-1]

ratios = sorted_std[:-1] / sorted_std[1:]

largest_gaps = np.argsort(ratios)[::-1][:15]

print("Largest gaps:")
for idx in largest_gaps:
    print(
        f"Between rank {idx+1} and {idx+2}: "
        f"{sorted_std[idx]:.8f} -> {sorted_std[idx+1]:.8f}, "
        f"ratio = {ratios[idx]:.2f}"
    )

In [ ]:
import numpy as np


std = np.std(Z, axis=0)
ranked_dims = np.argsort(std)[::-1]

for k in [20, 30, 40, 45, 46, 50, 55, 60, 80, 128]:

    selected_dims = ranked_dims[:k]

    Z_k = Z[:, selected_dims]

    print(f"k={k}, shape={Z_k.shape}")

In [ ]:
# ============================================================
# Top-variance latent dimensions: UMAP + t-SNE
# k = 36 ... 56
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap.umap_ as umap


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

LATENT_PATH = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_1M_4k/latent_codes.npy"

OUTPUT_DIR = "/home/ids/gmargari-24/airway_project/Data/latent_projections_36_56"

# Test every number of selected dimensions from 36 to 56
K_VALUES = range(36, 57)

# Standardize selected dimensions before UMAP / t-SNE
STANDARDIZE = True

# Random seed for reproducibility
RANDOM_STATE = 42


# ------------------------------------------------------------
# 2. Load latent codes
# ------------------------------------------------------------

Z = np.load(LATENT_PATH)

print("Latent shape:", Z.shape)

if Z.ndim != 2:
    raise ValueError(f"Expected a 2D latent matrix, got shape {Z.shape}")

n_samples, n_features = Z.shape

if n_features != 128:
    print(
        f"Warning: expected 128 latent dimensions, "
        f"but found {n_features}"
    )


# ------------------------------------------------------------
# 3. Rank dimensions according to STD
# ------------------------------------------------------------

std = np.std(Z, axis=0)

# Indices sorted from largest STD -> smallest STD
ranked_dims = np.argsort(std)[::-1]

print("\nTop 20 latent dimensions by STD:")
print("-" * 60)

for rank, dim in enumerate(ranked_dims[:20], start=1):
    print(
        f"Rank {rank:2d} | "
        f"Dimension {dim:3d} | "
        f"STD = {std[dim]:.8f}"
    )


# ------------------------------------------------------------
# 4. Create output directories
# ------------------------------------------------------------

umap_dir = os.path.join(OUTPUT_DIR, "UMAP")
tsne_dir = os.path.join(OUTPUT_DIR, "tSNE")

os.makedirs(umap_dir, exist_ok=True)
os.makedirs(tsne_dir, exist_ok=True)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ------------------------------------------------------------
# 5. Loop over number of selected dimensions
# ------------------------------------------------------------

for k in K_VALUES:

    print("\n" + "=" * 70)
    print(f"Processing top {k} variance dimensions")
    print("=" * 70)


    # --------------------------------------------------------
    # Select top-k highest variance dimensions
    # --------------------------------------------------------

    selected_dims = ranked_dims[:k]

    Z_k = Z[:, selected_dims]

    print("Selected representation shape:", Z_k.shape)

    print("Selected dimensions:")
    print(selected_dims.tolist())


    # --------------------------------------------------------
    # Optional standardization
    # --------------------------------------------------------

    if STANDARDIZE:

        scaler = StandardScaler()

        Z_k_processed = scaler.fit_transform(Z_k)

    else:

        Z_k_processed = Z_k.copy()


    # ========================================================
    # 6. UMAP
    # ========================================================

    print("\nRunning UMAP...")

    umap_model = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=RANDOM_STATE
    )

    umap_embedding = umap_model.fit_transform(Z_k_processed)

    print("UMAP shape:", umap_embedding.shape)


    # --------------------------------------------------------
    # Save UMAP embedding
    # --------------------------------------------------------

    umap_npy_path = os.path.join(
        umap_dir,
        f"umap_top{k}.npy"
    )

    np.save(
        umap_npy_path,
        umap_embedding
    )


    # --------------------------------------------------------
    # Plot UMAP
    # --------------------------------------------------------

    plt.figure(figsize=(8, 7))

    plt.scatter(
        umap_embedding[:, 0],
        umap_embedding[:, 1],
        s=35,
        alpha=0.8
    )

    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")

    plt.title(
        f"UMAP - Top {k} Variance Latent Dimensions"
    )

    plt.grid(alpha=0.25)

    plt.tight_layout()

    umap_png_path = os.path.join(
        umap_dir,
        f"umap_top{k}.png"
    )

    plt.savefig(
        umap_png_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


    # ========================================================
    # 7. t-SNE
    # ========================================================

    print("\nRunning t-SNE...")

    tsne_model = TSNE(
        n_components=2,
        perplexity=50,
        init="pca",
        random_state=RANDOM_STATE,
        max_iter=1000
    )

    tsne_embedding = tsne_model.fit_transform(
        Z_k_processed
    )

    print("t-SNE shape:", tsne_embedding.shape)


    # --------------------------------------------------------
    # Save t-SNE embedding
    # --------------------------------------------------------

    tsne_npy_path = os.path.join(
        tsne_dir,
        f"tsne_top{k}.npy"
    )

    np.save(
        tsne_npy_path,
        tsne_embedding
    )


    # --------------------------------------------------------
    # Plot t-SNE
    # --------------------------------------------------------

    plt.figure(figsize=(8, 7))

    plt.scatter(
        tsne_embedding[:, 0],
        tsne_embedding[:, 1],
        s=35,
        alpha=0.8
    )

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")

    plt.title(
        f"t-SNE - Top {k} Variance Latent Dimensions"
    )

    plt.grid(alpha=0.25)

    plt.tight_layout()

    tsne_png_path = os.path.join(
        tsne_dir,
        f"tsne_top{k}.png"
    )

    plt.savefig(
        tsne_png_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


    # --------------------------------------------------------
    # Save selected dimensions
    # --------------------------------------------------------

    dims_path = os.path.join(
        OUTPUT_DIR,
        f"selected_dims_top{k}.npy"
    )

    np.save(
        dims_path,
        selected_dims
    )

    print(f"\nFinished k={k}")


print("\n" + "=" * 70)
print("ALL PROJECTIONS COMPLETED")
print("=" * 70)

print(f"\nUMAP results: {umap_dir}")
print(f"t-SNE results: {tsne_dir}")

# 2. K-Means

In [ ]:
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# Load latent representation
# ------------------------------------------------------------

Z = np.load(
    "/home/ids/gmargari-24/airway_project/Data/"
    "latent_projections_36_56/selected_dims_top46.npy"
)

# IMPORTANT:
# selected_dims_top46.npy contains dimension indices,
# not the actual latent representation.

Z_all = np.load(
    "/home/ids/gmargari-24/airway_project/"
    "DGCI/logs/binary_128_1M_4k/latent_codes.npy"
)

Z = Z_all[:, Z]

print("Representation shape:", Z.shape)

# ------------------------------------------------------------
# Standardize
# ------------------------------------------------------------

Z_std = StandardScaler().fit_transform(Z)

# ------------------------------------------------------------
# Test K values
# ------------------------------------------------------------

results = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(Z_std)

    silhouette = silhouette_score(Z_std, labels)
    db = davies_bouldin_score(Z_std, labels)
    ch = calinski_harabasz_score(Z_std, labels)

    results.append(
        [k, silhouette, db, ch]
    )

    print(
        f"K={k:2d} | "
        f"Silhouette={silhouette:.4f} | "
        f"DB={db:.4f} | "
        f"CH={ch:.2f}"
    )

# 3. Plot the K-Means evaluation

In [ ]:
import matplotlib.pyplot as plt

results = np.array(results)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    results[:, 0],
    results[:, 1],
    marker="o"
)

ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Silhouette score")
ax.set_title("K-Means clustering evaluation")

ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## all the reuslts 

In [ ]:
# ============================================================
# CLUSTERING EXPERIMENT
#
# Top-variance latent dimensions: k = 36 ... 56
#
# Methods:
#   1. K-Means
#   2. Gaussian Mixture Model (GMM)
#   3. HDBSCAN
#
# Evaluation:
#   - Silhouette score
#   - Davies-Bouldin index
#   - Calinski-Harabasz index
#   - Inertia (K-Means)
#   - AIC / BIC (GMM)
#   - Number of clusters
#   - Noise percentage (HDBSCAN)
#
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# HDBSCAN
# ------------------------------------------------------------

try:
    import hdbscan
except ImportError:
    raise ImportError(
        "HDBSCAN is not installed.\n"
        "Install it with:\n"
        "pip install hdbscan"
    )


# ============================================================
# 1. PATHS
# ============================================================

LATENT_PATH = (
    "/home/ids/gmargari-24/airway_project/"
    "DGCI/logs/binary_128_1M_4k/latent_codes.npy"
)

OUTPUT_DIR = (
    "/home/ids/gmargari-24/airway_project/"
    "Data/clustering_36_56"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Subdirectories
# ------------------------------------------------------------

KMEANS_DIR = os.path.join(
    OUTPUT_DIR,
    "KMeans"
)

GMM_DIR = os.path.join(
    OUTPUT_DIR,
    "GMM"
)

HDBSCAN_DIR = os.path.join(
    OUTPUT_DIR,
    "HDBSCAN"
)

PLOTS_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

for directory in [
    KMEANS_DIR,
    GMM_DIR,
    HDBSCAN_DIR,
    PLOTS_DIR
]:
    os.makedirs(directory, exist_ok=True)


# ============================================================
# 2. PARAMETERS
# ============================================================

# Number of top-variance latent dimensions
DIMENSION_VALUES = range(36, 57)

# K-Means / GMM
CLUSTER_VALUES = range(2, 11)

# HDBSCAN
MIN_CLUSTER_SIZE_VALUES = [10, 15, 20, 25, 30]
MIN_SAMPLES_VALUES = [5, 10, 15]

RANDOM_STATE = 42

# Standardize selected latent dimensions
STANDARDIZE = True


# ============================================================
# 3. LOAD LATENT REPRESENTATION
# ============================================================

Z = np.load(LATENT_PATH)

print("=" * 70)
print("LATENT REPRESENTATION")
print("=" * 70)

print("Shape:", Z.shape)

if Z.ndim != 2:
    raise ValueError(
        f"Expected 2D array, got {Z.shape}"
    )

N, D = Z.shape

print("Number of samples:", N)
print("Number of dimensions:", D)


# ============================================================
# 4. RANK LATENT DIMENSIONS BY VARIANCE
# ============================================================

std = np.std(Z, axis=0)

# Largest STD -> smallest STD
ranked_dims = np.argsort(std)[::-1]

print("\nTop 20 dimensions by STD:")
print("-" * 70)

for rank, dim in enumerate(
    ranked_dims[:20],
    start=1
):
    print(
        f"Rank {rank:2d} | "
        f"Dim {dim:3d} | "
        f"STD={std[dim]:.8f}"
    )


# ============================================================
# 5. HELPER FUNCTION
# ============================================================

def calculate_metrics(
    X,
    labels,
    ignore_noise=False
):
    """
    Calculate clustering metrics.

    Returns:
        silhouette
        davies_bouldin
        calinski_harabasz
        n_clusters
        n_noise
        noise_percentage
    """

    labels = np.asarray(labels)

    # --------------------------------------------------------
    # Noise handling
    # --------------------------------------------------------

    if ignore_noise:
        valid_mask = labels != -1
    else:
        valid_mask = np.ones(
            len(labels),
            dtype=bool
        )

    X_eval = X[valid_mask]
    labels_eval = labels[valid_mask]

    n_noise = np.sum(labels == -1)

    noise_percentage = (
        100.0 * n_noise / len(labels)
    )

    # Number of clusters
    unique_clusters = np.unique(
        labels_eval
    )

    n_clusters = len(unique_clusters)

    # --------------------------------------------------------
    # Need at least two clusters
    # --------------------------------------------------------

    if n_clusters >= 2:

        silhouette = silhouette_score(
            X_eval,
            labels_eval
        )

        davies = davies_bouldin_score(
            X_eval,
            labels_eval
        )

        calinski = calinski_harabasz_score(
            X_eval,
            labels_eval
        )

    else:

        silhouette = np.nan
        davies = np.nan
        calinski = np.nan

    return {
        "silhouette": silhouette,
        "davies_bouldin": davies,
        "calinski_harabasz": calinski,
        "n_clusters": n_clusters,
        "n_noise": int(n_noise),
        "noise_percentage": noise_percentage
    }


# ============================================================
# 6. STORAGE FOR ALL RESULTS
# ============================================================

kmeans_results = []
gmm_results = []
hdbscan_results = []


# ============================================================
# 7. MAIN LOOP
# ============================================================

for k_dim in DIMENSION_VALUES:

    print("\n")
    print("#" * 80)
    print(
        f"PROCESSING TOP {k_dim} LATENT DIMENSIONS"
    )
    print("#" * 80)

    # --------------------------------------------------------
    # Select top-k dimensions
    # --------------------------------------------------------

    selected_dims = ranked_dims[:k_dim]

    X = Z[:, selected_dims]

    print(
        "Selected representation:",
        X.shape
    )

    print(
        "Dimension indices:",
        selected_dims.tolist()
    )

    # --------------------------------------------------------
    # Save selected dimension indices
    # --------------------------------------------------------

    np.save(
        os.path.join(
            OUTPUT_DIR,
            f"selected_dims_top{k_dim}.npy"
        ),
        selected_dims
    )

    # --------------------------------------------------------
    # Standardization
    # --------------------------------------------------------

    if STANDARDIZE:
        scaler = StandardScaler()
        X_processed = scaler.fit_transform(X)
    else:
        X_processed = X.copy()

    X_processed = X_processed.astype(np.float64)

    # Save the resulting representation
    np.save(
        os.path.join(
            OUTPUT_DIR,
            f"latent_top{k_dim}.npy"
        ),
        X_processed
    )


    # ========================================================
    # 7A. K-MEANS
    # ========================================================

    print("\n--- K-MEANS ---")

    for n_clusters in CLUSTER_VALUES:

        print(
            f"K-Means: dimensions={k_dim}, "
            f"K={n_clusters}"
        )

        model = KMeans(
            n_clusters=n_clusters,
            n_init=50,
            random_state=RANDOM_STATE
        )

        labels = model.fit_predict(
            X_processed
        )

        metrics = calculate_metrics(
            X_processed,
            labels
        )

        result = {
            "dimensions": k_dim,
            "n_clusters": n_clusters,
            "inertia": model.inertia_,
            **metrics
        }

        kmeans_results.append(result)

        # ----------------------------------------------------
        # Save labels
        # ----------------------------------------------------

        label_path = os.path.join(
            KMEANS_DIR,
            f"kmeans_top{k_dim}_K{n_clusters}.npy"
        )

        np.save(
            label_path,
            labels
        )


    # ========================================================
    # 7B. GMM
    # ========================================================



    print("\n--- GMM ---")

    for n_components in CLUSTER_VALUES:

        print(
            f"GMM: dimensions={k_dim}, "
            f"components={n_components}"
        )

        try:

            gmm = GaussianMixture(
                n_components=n_components,
                covariance_type="full",
                random_state=RANDOM_STATE,
                n_init=10,
                max_iter=1000,
                reg_covar=1e-6
            )

            labels = gmm.fit_predict(
                X_processed
            )

            metrics = calculate_metrics(
                X_processed,
                labels
            )

            result = {
                "dimensions": k_dim,
                "n_components": n_components,
                "AIC": gmm.aic(X_processed),
                "BIC": gmm.bic(X_processed),
                **metrics,
                "status": "success"
            }

            gmm_results.append(result)

            # ------------------------------------------------
            # Save labels
            # ------------------------------------------------

            label_path = os.path.join(
                GMM_DIR,
                f"gmm_top{k_dim}_K{n_components}.npy"
            )

            np.save(
                label_path,
                labels
            )

            print(
                f"    Success | "
                f"Silhouette={metrics['silhouette']:.4f} | "
                f"BIC={gmm.bic(X_processed):.2f}"
            )

        except Exception as e:

            print(
                f"    FAILED | "
                f"dimensions={k_dim}, "
                f"components={n_components}"
            )

            print(
                f"    Error: {type(e).__name__}: {e}"
            )

            # Save failed configuration to results
            gmm_results.append({
                "dimensions": k_dim,
                "n_components": n_components,
                "AIC": np.nan,
                "BIC": np.nan,
                "silhouette": np.nan,
                "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan,
                "n_clusters": np.nan,
                "n_noise": np.nan,
                "noise_percentage": np.nan,
                "status": "failed"
            })

            # Continue to the next GMM configuration
            continue

    # ========================================================
    # 7C. HDBSCAN
    # ========================================================

    print("\n--- HDBSCAN ---")

    for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:

        for min_samples in MIN_SAMPLES_VALUES:

            print(
                f"HDBSCAN: "
                f"dimensions={k_dim}, "
                f"min_cluster_size={min_cluster_size}, "
                f"min_samples={min_samples}"
            )

            model = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="euclidean",
                cluster_selection_method="eom",
                prediction_data=True
            )

            labels = model.fit_predict(
                X_processed
            )

            metrics = calculate_metrics(
                X_processed,
                labels,
                ignore_noise=True
            )

            # ------------------------------------------------
            # HDBSCAN cluster persistence
            # ------------------------------------------------

            if hasattr(
                model,
                "relative_validity_"
            ):
                relative_validity = (
                    model.relative_validity_
                )
            else:
                relative_validity = np.nan

            result = {
                "dimensions": k_dim,
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "relative_validity": relative_validity,
                **metrics
            }

            hdbscan_results.append(
                result
            )

            # ------------------------------------------------
            # Save labels
            # ------------------------------------------------

            label_path = os.path.join(
                HDBSCAN_DIR,
                (
                    f"hdbscan_top{k_dim}"
                    f"_mcs{min_cluster_size}"
                    f"_ms{min_samples}.npy"
                )
            )

            np.save(
                label_path,
                labels
            )


# ============================================================
# 8. CONVERT RESULTS TO DATAFRAMES
# ============================================================

kmeans_df = pd.DataFrame(
    kmeans_results
)

gmm_df = pd.DataFrame(
    gmm_results
)

hdbscan_df = pd.DataFrame(
    hdbscan_results
)


# ============================================================
# 9. SAVE ALL RESULTS
# ============================================================

kmeans_csv = os.path.join(
    OUTPUT_DIR,
    "kmeans_results.csv"
)

gmm_csv = os.path.join(
    OUTPUT_DIR,
    "gmm_results.csv"
)

hdbscan_csv = os.path.join(
    OUTPUT_DIR,
    "hdbscan_results.csv"
)

kmeans_df.to_csv(
    kmeans_csv,
    index=False
)

gmm_df.to_csv(
    gmm_csv,
    index=False
)

hdbscan_df.to_csv(
    hdbscan_csv,
    index=False
)


# ============================================================
# 10. FIND BEST K-MEANS CONFIGURATION
# ============================================================

best_kmeans = (
    kmeans_df
    .sort_values(
        "silhouette",
        ascending=False
    )
    .groupby("dimensions")
    .first()
    .reset_index()
)

best_kmeans.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_kmeans_per_dimension.csv"
    ),
    index=False
)


# ============================================================
# 11. FIND BEST GMM CONFIGURATION
# ============================================================

best_gmm_silhouette = (
    gmm_df
    .sort_values(
        "silhouette",
        ascending=False
    )
    .groupby("dimensions")
    .first()
    .reset_index()
)

best_gmm_silhouette.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_gmm_per_dimension_by_silhouette.csv"
    ),
    index=False
)


# ============================================================
# 12. FIND BEST HDBSCAN CONFIGURATION
# ============================================================

best_hdbscan = (
    hdbscan_df
    .sort_values(
        "silhouette",
        ascending=False
    )
    .groupby("dimensions")
    .first()
    .reset_index()
)

best_hdbscan.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_hdbscan_per_dimension.csv"
    ),
    index=False
)


# ============================================================
# 13. SUMMARY PLOT: K-MEANS
# ============================================================

plt.figure(figsize=(10, 6))

for k_dim in DIMENSION_VALUES:

    subset = kmeans_df[
        kmeans_df["dimensions"] == k_dim
    ]

    plt.plot(
        subset["n_clusters"],
        subset["silhouette"],
        marker="o",
        label=f"{k_dim}D"
    )

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title(
    "K-Means Silhouette Score\n"
    "for Top-Variance Latent Dimensions"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        PLOTS_DIR,
        "kmeans_silhouette_all_dimensions.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 14. SUMMARY: BEST K-MEANS SILHOUETTE vs DIMENSIONS
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    best_kmeans["dimensions"],
    best_kmeans["silhouette"],
    marker="o"
)

plt.xlabel(
    "Number of selected latent dimensions"
)

plt.ylabel(
    "Best Silhouette score"
)

plt.title(
    "Best K-Means Clustering Quality "
    "vs Latent Dimensionality"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        PLOTS_DIR,
        "kmeans_best_silhouette_vs_dimensions.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 15. SUMMARY: GMM
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    best_gmm_silhouette["dimensions"],
    best_gmm_silhouette["silhouette"],
    marker="o"
)

plt.xlabel(
    "Number of selected latent dimensions"
)

plt.ylabel(
    "Best Silhouette score"
)

plt.title(
    "Best GMM Silhouette Score "
    "vs Latent Dimensionality"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        PLOTS_DIR,
        "gmm_best_silhouette_vs_dimensions.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 16. SUMMARY: HDBSCAN
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    best_hdbscan["dimensions"],
    best_hdbscan["silhouette"],
    marker="o"
)

plt.xlabel(
    "Number of selected latent dimensions"
)

plt.ylabel(
    "Best Silhouette score"
)

plt.title(
    "Best HDBSCAN Silhouette Score "
    "vs Latent Dimensionality"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        PLOTS_DIR,
        "hdbscan_best_silhouette_vs_dimensions.png"
    ),
    dpi=300
)

plt.close()


# ============================================================
# 17. FINAL SUMMARY TABLE
# ============================================================

summary = pd.DataFrame({
    "dimensions": list(DIMENSION_VALUES),

    "kmeans_best_silhouette": [
        best_kmeans.loc[
            best_kmeans["dimensions"] == d,
            "silhouette"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "kmeans_best_K": [
        best_kmeans.loc[
            best_kmeans["dimensions"] == d,
            "n_clusters"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "gmm_best_silhouette": [
        best_gmm_silhouette.loc[
            best_gmm_silhouette["dimensions"] == d,
            "silhouette"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "gmm_best_components": [
        best_gmm_silhouette.loc[
            best_gmm_silhouette["dimensions"] == d,
            "n_components"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "hdbscan_best_silhouette": [
        best_hdbscan.loc[
            best_hdbscan["dimensions"] == d,
            "silhouette"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "hdbscan_best_clusters": [
        best_hdbscan.loc[
            best_hdbscan["dimensions"] == d,
            "n_clusters"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ],

    "hdbscan_best_noise_percentage": [
        best_hdbscan.loc[
            best_hdbscan["dimensions"] == d,
            "noise_percentage"
        ].iloc[0]
        for d in DIMENSION_VALUES
    ]
})

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "FINAL_CLUSTERING_SUMMARY.csv"
    ),
    index=False
)


# ============================================================
# 18. PRINT FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("FINAL CLUSTERING SUMMARY")
print("=" * 80)

print(
    summary.to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("EXPERIMENT COMPLETED")
print("=" * 80)

print(
    "\nResults saved to:"
)

print(OUTPUT_DIR)